## Import Libraries 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

# Display all outputs from each cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# formatting the output
pd.options.display.float_format= '{:.2f}'.format

#   Connect to Database System 

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
server = "YOUR_SQL_SERVER"  # Replace with your SQL Server instance
database = "University_DB"     # your database name

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)


# 1 - Ingests all CSVs, enforces types, validates FK integrity, checks rules

In [ ]:

departments = pd.read_sql("SELECT * FROM departments", engine)
majors = pd.read_sql("SELECT * FROM majors", engine)
professors = pd.read_sql("SELECT * FROM professors", engine)
students = pd.read_sql("SELECT * FROM students", engine)
courses = pd.read_sql("SELECT * FROM courses", engine)
prerequisites = pd.read_sql("SELECT * FROM prerequisites", engine)
classrooms = pd.read_sql("SELECT * FROM classrooms", engine)
timeslots = pd.read_sql("SELECT * FROM timeslots", engine)
course_offerings = pd.read_sql("SELECT * FROM course_offerings", engine)
offering_timeslots = pd.read_sql("SELECT * FROM offering_timeslots", engine)
enrollments = pd.read_sql("SELECT * FROM enrollments", engine)
assessment_components = pd.read_sql("SELECT * FROM assessment_components", engine)
student_component_scores = pd.read_sql("SELECT * FROM student_component_scores", engine)
evaluations = pd.read_sql("SELECT * FROM evaluations", engine)
evaluation_summary = pd.read_sql("SELECT * FROM evaluation_summary", engine)


In [ ]:
tables = {
    'departments': departments,
    'majors': majors,
    'professors': professors,
    'students': students,
    'courses': courses,
    'prerequisites': prerequisites,
    'classrooms': classrooms,
    'timeslots': timeslots,
    'course_offerings': course_offerings,
    'offering_timeslots': offering_timeslots,
    'enrollments': enrollments,
    'assessment_components': assessment_components,
    'student_component_scores': student_component_scores,
    'evaluations': evaluations,
    'evaluation_summary': evaluation_summary
}

In [ ]:
summary = []
for name, df in tables.items():
    summary.append({
        'table': name,
        'rows': len(df),
        'cols': df.shape[1],
        'null_percent_sample %': (df.isnull().mean().max() * 100).round(2),
        'dtypes': ', '.join([f"{c}:{str(t)}" for c,t in df.dtypes.items()][:5]) + ('...' if df.shape[1]>5 else '')
    })
pd.DataFrame(summary).set_index('table')


In [ ]:
expected_types = {
    'students': { 'dob': 'datetime64[ns]' },
    'enrollments': { 'final_score': 'float64', 'gpa_points': 'float64'},
    'assessment_components': { 'weight_pct': 'float64', 'max_points': 'float64'}

}

def coerce_types(df, mapping):
    for col, dtype in mapping.items():
        if col in df.columns:
            try:
                if 'datetime' in dtype:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                elif dtype.startswith('float'):
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
                else:
                    df[col] = df[col].astype(dtype)
            except Exception as e:
                print(f"Warning: could not coerce {col} in mapping: {e}")
    return df


for tname, mapping in expected_types.items():
    if tname in tables:
        tables[tname] = coerce_types(tables[tname], mapping)


for name in expected_types.keys():
    if name in tables:
        print(f"\n{name} dtypes:")
        print(tables[name].dtypes)


In [ ]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"\n🔹 {name} ({len(df)} rows):")
        print(nulls.sort_values(ascending=False))


In [ ]:

fks = [
    ('enrollments', 'student_id', 'students', 'student_id'),
    ('enrollments', 'offering_id', 'course_offerings', 'offering_id'),
    ('assessment_components', 'offering_id', 'course_offerings', 'offering_id'),
    ('student_component_scores', 'student_id', 'students', 'student_id'),
    ('student_component_scores', 'component_id', 'assessment_components', 'component_id'),
    ('course_offerings', 'course_id', 'courses', 'course_id'),
    ('course_offerings', 'professor_id', 'professors', 'professor_id'),
    ('prerequisites', 'prereq_course_id', 'courses', 'course_id'),
    ('offering_timeslots', 'timeslot_id', 'timeslots', 'timeslot_id'),
    ('offering_timeslots', 'classroom_id', 'classrooms', 'classroom_id')
]

fk_issues = []
for child, child_col, parent, parent_col in fks:
    if child in tables and parent in tables:
        missing = tables[child][~tables[child][child_col].isin(tables[parent][parent_col])]
        cnt = len(missing)
        fk_issues.append({'child_table': child, 'child_col': child_col, 'parent_table': parent, 'parent_col': parent_col, 'missing_count': cnt})
        if cnt>0:
            print(f"\nFK issue: {child}.{child_col} -> {parent}.{parent_col}  MISSING: {cnt}")
            display(missing.head(10))

pd.DataFrame(fk_issues)


In [ ]:

# Rule A: sum weight_pct per offering = 100
weights = tables['assessment_components'].groupby('offering_id')['weight_pct'].sum().reset_index()
bad_weights = weights[weights['weight_pct'].round(3) != 100.0]  
print("Offerings with bad weight sum:")
display(bad_weights)

# Rule B: final_score between 0 and 100
bad_scores = tables['enrollments'][(tables['enrollments']['final_score'] < 0) | (tables['enrollments']['final_score'] > 100)]
print("Out-of-range final_score count:", len(bad_scores))
display(bad_scores.head(10))

# Rule C: gpa_points between 0 and 4
bad_gpa = tables['enrollments'][(tables['enrollments']['gpa_points'] < 0) | (tables['enrollments']['gpa_points'] > 4)]
print("Out-of-range gpa_points count:", len(bad_gpa))
display(bad_gpa.head(10))

# Rule D: grades allowed values (A,B,C,D,F)
allowed_grades = {'A','B','C','D','F'}
bad_grades = tables['enrollments'][~tables['enrollments']['grade'].isin(allowed_grades) & tables['enrollments']['grade'].notnull()]
print("Invalid grade values:", len(bad_grades))
display(bad_grades.head(10))


In [ ]:
def map_score_to_grade_gpa(score):
    if pd.isna(score):
        return pd.Series({'grade_new': None, 'gpa_new': None})
    elif score >= 90:
        return pd.Series({'grade_new': 'A', 'gpa_new': 4.0})
    elif score >= 80:
        return pd.Series({'grade_new': 'B', 'gpa_new': 3.0})
    elif score >= 70:
        return pd.Series({'grade_new': 'C', 'gpa_new': 2.0})
    elif score >= 60:
        return pd.Series({'grade_new': 'D', 'gpa_new': 1.0})
    else:
        return pd.Series({'grade_new': 'F', 'gpa_new': 0.0})


In [ ]:
mapped = enrollments['final_score'].apply(map_score_to_grade_gpa)
enrollments = pd.concat([enrollments, mapped], axis=1)


In [ ]:

mask_gpa = (enrollments['gpa_points'] < 0) | (enrollments['gpa_points'] > 4) | (enrollments['gpa_points'].isna())
enrollments.loc[mask_gpa, 'gpa_points'] = enrollments.loc[mask_gpa, 'gpa_new']

mask_grade = ~enrollments['grade'].isin(['A', 'B', 'C', 'D', 'F']) | enrollments['grade'].isna()
enrollments.loc[mask_grade, 'grade'] = enrollments.loc[mask_grade, 'grade_new']


In [ ]:
enrollments.drop(columns=['grade_new', 'gpa_new'], inplace=True)

print(enrollments['gpa_points'].describe())
print(enrollments['grade'].value_counts())


In [ ]:
enrollments['gpa_points']

# 2 - Feature Engineering

## A - Compute per-enrollment component percentages and weighted final_score from components

In [ ]:
components_data = pd.read_sql("""
    SELECT 
        scs.student_id,
        scs.offering_id, 
        scs.component_id,
        scs.score_points,
        ac.max_points,
        ac.weight_pct
    FROM student_component_scores scs
    JOIN assessment_components ac 
        ON scs.component_id = ac.component_id 
        AND scs.offering_id = ac.offering_id
""", engine)

In [ ]:
components_data['component_percentage'] = (
    components_data['score_points'] / components_data['max_points'] * 100
)


components_data['weighted_contribution'] = (
    components_data['component_percentage'] * components_data['weight_pct'] / 100
)

python_calculated_scores = components_data.groupby(['student_id', 'offering_id']).agg({
    'weighted_contribution': 'sum'
}).reset_index()

python_calculated_scores = python_calculated_scores.rename(
    columns={'weighted_contribution': 'python_final_score'}
)


comparison = enrollments[['student_id', 'offering_id', 'final_score']].merge(
    python_calculated_scores, 
    on=['student_id', 'offering_id'], 
    how='left'
)

comparison['score_difference'] = abs(comparison['final_score'] - comparison['python_final_score'])



In [ ]:
comparison

## B - Build offering-level aggregates: average final_score, grade distribution, pass rate, utilization rate (enrolled/capacity).

In [ ]:
offering_level = enrollments.groupby('offering_id').agg(
    avg_final_score=('final_score', 'mean'),
    pass_rate=('grade', lambda x: (x.isin(['A','B','C','D']).mean()) * 100)
).reset_index()

offering_level = offering_level.merge(course_offerings[['offering_id', 'capacity']], on='offering_id', how='left')
enroll_count = enrollments.groupby('offering_id')['student_id'].count().reset_index(name='n_enrolled')
offering_level = offering_level.merge(enroll_count, on='offering_id', how='left')

offering_level['utilization_rate'] = (offering_level['n_enrolled'] / offering_level['capacity']) * 100



In [ ]:
offering_level

## C - Create a “Student Profile”

In [ ]:
students = """
SELECT
    s.student_id,
    s.gender,
    m.name AS major,
    s.start_year,
    c.credits,
    e.final_score, 
    e.gpa_points 
FROM
    students s
    INNER JOIN majors m ON s.major_id = m.major_id
    INNER JOIN enrollments e ON s.student_id = e.student_id
    INNER JOIN course_offerings co ON e.offering_id = co.offering_id
    INNER JOIN courses c ON co.course_id = c.course_id
ORDER BY
    s.start_year,
    s.student_id;
"""
df_students = pd.read_sql(students, engine)

In [ ]:
df_students

In [ ]:
df_students.info()
df_students.isnull().sum()

In [ ]:
df_students['final_score'] = df_students['final_score'].fillna(df_students['final_score'].mean()).round(2)
df_students['gpa_points'] = df_students['gpa_points'].fillna(df_students['gpa_points'].mean()).round(2)

In [ ]:
student_profile = df_students.groupby(['student_id', 'major', 'start_year'], as_index=False).agg(
    total_credits_attempted=('credits', 'sum'),
    mean_final_score=('final_score', 'mean'),
    cumulative_GPA=('gpa_points', lambda x: (x * df_students.loc[x.index, 'credits']).sum() / df_students.loc[x.index, 'credits'].sum())
)
student_profile['mean_final_score'] = student_profile['mean_final_score'].round(2)
student_profile['cumulative_GPA'] = student_profile['cumulative_GPA'].round(2)

In [ ]:
student_profile

# 3 - Analysis visuals

## A - Histograms of final_score, GPA

In [ ]:
plt.figure(figsize=(12, 5))


plt.subplot(1, 2, 1)
plt.hist(enrollments['final_score'].dropna(), bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Final Scores')
plt.xlabel('Final Score (0-100)')
plt.ylabel('Number of Students')
plt.grid(axis='y', linestyle='--', alpha=0.7)


plt.subplot(1, 2, 2)
plt.hist(student_profile['cumulative_GPA'].dropna(), bins=20, color='lightgreen', edgecolor='black')
plt.title('Distribution of Cumulative GPA')
plt.xlabel('GPA (0-4)')
plt.ylabel('Number of Students')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show();



## B - Bar charts: avg GPA by major; response rate by course/term; capacity utilization by building

### Average GPA by Major

In [ ]:
merged = enrollments.merge(tables['students'][['student_id', 'major_id']], on='student_id', how='left')
merged = merged.merge(majors[['major_id', 'name']], on='major_id', how='left')

In [ ]:

avg_gpa_major = merged.groupby('name')['gpa_points'].mean().reset_index()

avg_gpa_major = avg_gpa_major.sort_values('gpa_points', ascending=True).reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.barh(avg_gpa_major['name'], avg_gpa_major['gpa_points'], color='steelblue')
plt.title('Average GPA by Major (Sorted)')
plt.xlabel('Average GPA')
plt.ylabel('Major')
plt.grid(True, axis='x')
plt.tight_layout()
plt.show();

### Response Rate by Course

In [ ]:
eval_rate = evaluation_summary.merge(
    course_offerings[['offering_id', 'course_id', 'semester', 'year']],
    on='offering_id',
    how='left'
)
eval_rate = eval_rate.merge(
    courses[['course_id', 'course_name']],
    on='course_id',
    how='left'
)


resp_by_course = (
    eval_rate.groupby('course_name')['resp_rate']
    .mean()
    .reset_index()
    .sort_values('resp_rate', ascending=False)
)


plt.figure(figsize=(10,5))
plt.barh(resp_by_course['course_name'].head(20),
         resp_by_course['resp_rate'].head(20),
         color='lightgreen', edgecolor='black')

plt.title('Average Evaluation Response Rate by Course')
plt.xlabel('Response Rate (%)')
plt.ylabel('Course')
plt.gca().invert_yaxis()  
plt.tight_layout()
plt.show();


### Capacity Utilization by Building

In [ ]:

enrolled_count = enrollments.groupby('offering_id')['student_id'].count().reset_index(name='enrolled')

util = course_offerings.merge(enrolled_count, on='offering_id', how='left')
util = util.merge(offering_timeslots[['offering_id', 'classroom_id']], on='offering_id', how='left')
util = util.merge(classrooms[['classroom_id', 'building', 'capacity']], on='classroom_id', how='left')

util['utilization_rate'] = (util['enrolled'] / util['capacity_y']) * 100

# Compute average utilization rate per building
avg_util_by_building = util.groupby('building')['utilization_rate'].mean().reset_index().sort_values('utilization_rate', ascending=False)

plt.figure(figsize=(8,5))
plt.bar(avg_util_by_building['building'], avg_util_by_building['utilization_rate'], color='orange', edgecolor='black')
plt.title('Average Capacity Utilization by Building')
plt.xlabel('Building')
plt.ylabel('Utilization (%)')
plt.tight_layout()
plt.show();

## C - Relationship plots: avg_overall (evals) vs avg_final_score (offerings)

In [ ]:
offer_avg = enrollments.groupby('offering_id')['final_score'].mean().reset_index()
offer_avg.rename(columns={'final_score': 'avg_final_score'}, inplace=True)

rel_data = evaluation_summary.merge(offer_avg, on='offering_id', how='left')


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(rel_data['avg_overall'], rel_data['avg_final_score'],
            color='skyblue', edgecolor='black', alpha=0.7)

plt.title('Relationship: Evaluation Score vs Final Score')
plt.xlabel('Average Evaluation Overall (1–5)')
plt.ylabel('Average Final Score (0–100)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show();


In [ ]:
correlation = rel_data['avg_overall'].corr(rel_data['avg_final_score'])
print(f"Correlation: {correlation:.2f}")


# 4 - Reconciliation report

In [ ]:
eval_sql = pd.read_sql("""
SELECT 
    offering_id,
    AVG(q_overall) as avg_overall,
    AVG(q_clarity) as avg_clarity,
    AVG(q_organization) as avg_organization,
    AVG(q_engagement) as avg_engagement,
    AVG(q_feedback) as avg_feedback,
    AVG(q_difficulty) as avg_difficulty
FROM evaluations
GROUP BY offering_id
ORDER BY offering_id;
""", engine)

In [ ]:
recomputed_eval_py = evaluations.groupby('offering_id').agg({
	'q_overall': 'mean',
	'q_clarity': 'mean',
	'q_organization': 'mean',
	'q_engagement': 'mean',
	'q_feedback': 'mean',
	'q_difficulty': 'mean'
}).reset_index()

# Optionally, rename columns for consistency with other DataFrames
recomputed_eval_py = recomputed_eval_py.rename(columns={
	'q_overall': 'avg_overall_py',
	'q_clarity': 'avg_clarity_py',
	'q_organization': 'avg_organization_py',
	'q_engagement': 'avg_engagement_py',
	'q_feedback': 'avg_feedback_py',
	'q_difficulty': 'avg_difficulty_py'
})

In [ ]:
eval_summary_sql = eval_sql.rename(columns={
    'avg_overall': 'avg_overall_sql',
    'avg_clarity': 'avg_clarity_sql', 
    'avg_organization': 'avg_organization_sql',
    'avg_engagement': 'avg_engagement_sql',
    'avg_feedback': 'avg_feedback_sql',
    'avg_difficulty': 'avg_difficulty_sql'
})

recomputed_eval = recomputed_eval_py.rename(columns={
    'avg_overall.py': 'avg_overall_py',
    'avg_clarity.py': 'avg_clarity_py',
    'avg_organization.py': 'avg_organization_py', 
    'avg_engagement.py': 'avg_engagement_py',
    'avg_feedback.py': 'avg_feedback_py',
    'avg_difficulty.py': 'avg_difficulty_py'
})

evaluation_summary_orig = evaluation_summary.rename(columns={
    'avg_overall': 'avg_overall_orig',
    'avg_clarity': 'avg_clarity_orig',
    'avg_organization': 'avg_organization_orig',
    'avg_engagement': 'avg_engagement_orig', 
    'avg_feedback': 'avg_feedback_orig',
    'avg_difficulty': 'avg_difficulty_orig'
})

# merge all three sources

three_way_comparison = eval_summary_sql.merge(
    recomputed_eval, on='offering_id', how='inner'
).merge(
    evaluation_summary_orig, on='offering_id', how='inner'
)


In [ ]:
eval_summary_sql
recomputed_eval
evaluation_summary_orig

In [ ]:
metrics = ['overall', 'clarity', 'organization', 'engagement', 'feedback', 'difficulty']


comparison_pairs = [
    ('py', 'sql', 'Python vs SQL'),
    ('py', 'orig', 'Python vs Original'), 
    ('sql', 'orig', 'SQL vs Original')
]

for source1, source2, label in comparison_pairs:
    print(f"\n   🔄 {label}:")
    print("   " + "-" * 40)
    
    for metric in metrics:
        col1 = f'avg_{metric}_{source1}'
        col2 = f'avg_{metric}_{source2}'
        
        diff = abs(three_way_comparison[col1] - three_way_comparison[col2])
        avg_diff = diff.mean()
        max_diff = diff.max()
        mismatches = len(diff[diff > 0.01])
        
        print(f"   • {metric.upper():12} | Average differences: {avg_diff:.3f} | Maximum difference: {max_diff:.3f} | Mismatches: {mismatches}/{len(three_way_comparison)}")

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, metric in enumerate(metrics):
    # رسم المقارنة الثلاثية
    axes[i].scatter(three_way_comparison['avg_overall_orig'], 
                   three_way_comparison[f'avg_{metric}_py'], 
                   alpha=0.6, label='Python', color='blue', s=50)
    axes[i].scatter(three_way_comparison['avg_overall_orig'], 
                   three_way_comparison[f'avg_{metric}_sql'], 
                   alpha=0.6, label='SQL', color='red', s=50)
    axes[i].scatter(three_way_comparison['avg_overall_orig'], 
                   three_way_comparison[f'avg_{metric}_orig'], 
                   alpha=0.6, label='Original', color='green', s=50, marker='^')
    
    axes[i].set_title(f'{metric.upper()} - Comparison', fontweight='bold')
    axes[i].set_xlabel('Original Overall Score')
    axes[i].set_ylabel(f'{metric} Score')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show();